## Lecture 9 - Conditional Manatees

McElreath's lecture for today: https://www.youtube.com/watch?v=QhHfo6-Bx8o

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/statrethinking_winter2019

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(ggplot2)     # bayesplot figures are ggplots, so we can add to them
library(loo)         # WAIC and PSIS-LOO cross-validation

# Helper functions
stdize <- function(x) (x - mean(x))/sd(x)

indexall <- function(L){
    poo <- unique(L)
    Ix <- as.integer(factor(L, levels = poo))
    list(poo, Ix)
}

# Stan's sampler with the same defaults as pm.sample(1000): 4 chains, 1000 warmup, 1000 draws
sample_stan <- function(model, data){
    model$sample(data = data, chains = 4, parallel_chains = 4, iter_warmup = 1000,
                 iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)
}

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

## Information-based model comparsion

With the information criteria outlined last week, we can now see how to use this information in practice to help diagnose problems in model specification. This is a really powerful result, so we'll take our time to step through it.

# The Monkies

We'll illustrate how to do this using the primate data set

In [ ]:
mdata <- read.csv('Primates301.csv')
mdata$Spp <- paste(mdata$genus, mdata$species)
head(mdata)

In [ ]:
# Drop rows missing longevity, brain size, or body mass
mdata <- mdata[!is.na(mdata$longevity) & !is.na(mdata$brain) & !is.na(mdata$body), ]
dim(mdata)

In this situation we want to bulid a model looking at the influence of body mass and brain size on lifespan. We can represent the causal model using the `dagitty` package:

In [ ]:
library(dagitty)

In [ ]:
# Create Monkey DAG
mDAG <- dagitty("dag {
  M -> L
  M -> B
  B -> L
}")
coordinates(mDAG) <- list(x = c(M = 0, B = 1, L = 2), y = c(M = 0, B = -1, L = 0))

In [ ]:
# Plot DAG
plot(mDAG)
# savefig('primateDAG.jpg')

Here we're asserting that longer lifespans are caused by being bigger and having bigger brains (clever monkies). We can run three models to see which has the most WAIC-based support:

$$
M_{MB}: \enspace \enspace L \sim N(\beta_0+\beta_M M+\beta_B B, \sigma) \enspace \enspace
$$

$$
M_{M}: \enspace \enspace   L \sim N(\beta_0+\beta_M M, \sigma)
$$

$$
M_{B}: \enspace \enspace  L \sim N(\beta_0+\beta_B B, \sigma)
$$

Looking piecewise at how much better things are when adding each parameter.

In [ ]:
# Grab data

# Body mass
M <- stdize(log(mdata$body))
# Brain size
B <- stdize(log(mdata$brain))
# Lifespan
L <- stdize(log(mdata$longevity))
# Ratio of brain size to body size
R <- B/M

In [ ]:
mdata$body

In [ ]:
# Full monkey
M_MB_code <- "
data {
  int<lower=0> N;
  vector[N] BodyMass;
  vector[N] BrainSize;
  vector[N] Lifespan;
}
parameters {
  real Intercept;                    // Baseline intercept
  real M;                            // Body mass effect
  real B;                            // Brain size
  real<lower=0, upper=10> SD_obs;    // Error
}
transformed parameters {
  // Linear model
  vector[N] mu = Intercept + M*BodyMass + B*BrainSize;
}
model {
  // Priors
  Intercept ~ normal(0, 0.2);
  M ~ normal(0, 0.5);
  B ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);

  // Likelihood
  Lifespan ~ normal(mu, SD_obs);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = normal_lpdf(Lifespan[i] | mu[i], SD_obs);
}
"
M_MB <- cmdstan_model(write_stan_file(M_MB_code))

In [ ]:
# Run sampler
trace_mb <- sample_stan(M_MB, list(N = length(L), BodyMass = M, BrainSize = B, Lifespan = L))

In [ ]:
# Big monkey
M_M_code <- "
data {
  int<lower=0> N;
  vector[N] BodyMass;
  vector[N] Lifespan;
}
parameters {
  real Intercept;                    // Baseline intercept
  real M;                            // Body mass effect
  real<lower=0, upper=10> SD_obs;    // Error
}
transformed parameters {
  // Linear model
  vector[N] mu = Intercept + M*BodyMass;
}
model {
  // Priors
  Intercept ~ normal(0, 0.2);
  M ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);

  // Likelihood
  Lifespan ~ normal(mu, SD_obs);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = normal_lpdf(Lifespan[i] | mu[i], SD_obs);
}
"
M_M <- cmdstan_model(write_stan_file(M_M_code))

In [ ]:
# Run sampler
trace_m <- sample_stan(M_M, list(N = length(L), BodyMass = M, Lifespan = L))

In [ ]:
# Smart monkey
M_B_code <- "
data {
  int<lower=0> N;
  vector[N] BrainSize;
  vector[N] Lifespan;
}
parameters {
  real Intercept;                    // Baseline intercept
  real B;                            // Brain size
  real<lower=0, upper=10> SD_obs;    // Error
}
transformed parameters {
  // Linear model
  vector[N] mu = Intercept + B*BrainSize;
}
model {
  // Priors
  Intercept ~ normal(0, 0.2);
  B ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);

  // Likelihood
  Lifespan ~ normal(mu, SD_obs);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = normal_lpdf(Lifespan[i] | mu[i], SD_obs);
}
"
M_B <- cmdstan_model(write_stan_file(M_B_code))

In [ ]:
# Run sampler
trace_b <- sample_stan(M_B, list(N = length(L), BrainSize = B, Lifespan = L))

In [ ]:
# WAIC for each model, from the pointwise log-likelihood stored in generated quantities
WAIC_mb <- waic(trace_mb$draws("log_lik", format = "matrix"))
WAIC_m <- waic(trace_m$draws("log_lik", format = "matrix"))
WAIC_b <- waic(trace_b$draws("log_lik", format = "matrix"))

# Pointwise elpd for each model, which is what the pseudo-BMA weights are built from
lpd_point <- cbind(Full = WAIC_mb$pointwise[, "elpd_waic"],
                   Big = WAIC_m$pointwise[, "elpd_waic"],
                   Smart = WAIC_b$pointwise[, "elpd_waic"])
Wts <- as.numeric(pseudobma_weights(lpd_point, BB = TRUE))
names(Wts) <- colnames(lpd_point)

WAIC_compare <- loo_compare(list(Full = WAIC_mb, Big = WAIC_m, Smart = WAIC_b))
cbind(round(WAIC_compare[, c("elpd_diff", "se_diff", "elpd_waic", "p_waic", "waic", "se_waic")], 2),
      weight = round(Wts[rownames(WAIC_compare)], 2))

This table gives lots of useful information for comparision - but also relative WAIC based model weight - which has the interpretation of the WAIC-based probability that Full is the best model (lowest KL-divergence) in the set of models compared. The calcualtion for this is

$$
weight_{i} = \frac{exp(-\frac{1}{2}\Delta_{i})}{\sum_{i=1}^{K}-\frac{1}{2}\Delta_{k})}
$$

where $\Delta_{i}$ is the `elpd_diff` above, the difference in WAIC units between each model and the lowest WAIC model. One thing to watch: `loo_compare()` reports `elpd_diff` on the **elpd** (log-score) scale, where higher is better, so the best model sits at 0 and every other model is negative - the opposite sign to the deviance-scale differences. On that scale $-\frac{1}{2}\Delta_{i}$ *is* `elpd_diff`, so the weight calculation is just the exponential of the column as it stands. From the table above we can use the `elpd_diff` values to do the calculation:

In [ ]:
Wx <- exp(WAIC_compare[, "elpd_diff"])
round(Wx/sum(Wx), 2)

These are not identical as `loo` uses some other calculation but close enough...

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)

# The equivalent of az.plot_compare(): WAIC per model with its standard error
plot_compare <- function(cmp){
    mods <- rownames(cmp)
    yy <- length(mods):1
    wc <- cmp[, "waic"]
    sew <- cmp[, "se_waic"]
    # In-sample deviance, i.e. WAIC with the overfitting penalty taken back off
    insamp <- wc - 2*cmp[, "p_waic"]
    # Difference from the best model, on the deviance scale
    dse <- 2*cmp[, "se_diff"]
    xr <- range(c(wc - sew, wc + sew, insamp, wc + dse))
    par(mar = c(5, 7, 2, 2))
    plot(wc, yy, xlim = xr + c(-0.05, 0.05)*diff(xr), ylim = c(0.15, length(mods) + 0.5),
         yaxt = 'n', pch = 16, cex = 1.5, col = 'dodgerblue',
         xlab = 'Deviance (WAIC)', ylab = '')
    axis(2, at = yy, labels = mods, las = 1)
    segments(wc - sew, yy, wc + sew, yy, col = 'dodgerblue', lwd = 2)
    points(insamp, yy, pch = 2)
    abline(v = wc[1], lty = 2, col = 'grey40')
    segments(wc[-1] - dse[-1], yy[-1] + 0.2, wc[-1] + dse[-1], yy[-1] + 0.2, col = 'grey40', lwd = 2)
    points(wc[-1], yy[-1] + 0.2, pch = 16, col = 'grey40')
    legend("bottomright", legend = c('WAIC', 'In-sample deviance', 'Difference from best'),
           pch = c(16, 2, 16), col = c('dodgerblue', 'black', 'grey40'), bty = 'n', cex = 0.8)
    par(mar = c(5, 4, 4, 2) + 0.1)
}
plot_compare(WAIC_compare)
# savefig('waiccompare.jpg', width = 8, height = 4)

So if we look at the WAIC model results, we get evidence that the full model (M_MB) and the smart (M_B) model have equal support, while the body mass model is far worse. What's going on? Well to figure it out let's look at the posteriors from all three models:

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)

# Pull a model's draws out with the model name pasted onto each parameter
grab <- function(trace, model, pars){
    out <- trace$draws(pars, format = "matrix")
    colnames(out) <- paste0(model, ': ', pars)
    out
}
forest_draws <- cbind(grab(trace_mb, 'Full', c("Intercept", "M", "B", "SD_obs")),
                      grab(trace_m, 'Big', c("Intercept", "M", "SD_obs")),
                      grab(trace_b, 'Smart', c("Intercept", "B", "SD_obs")))

mcmc_intervals(forest_draws) + vline_0()
# ggsave('forestape.jpg', dpi = 300)

What you can see is that the effect of body size that is so strong in model `M_M` ('Big') gets negative in the full model (`M_MB`), while the effect of brain size in model `M_B` ('Smart') remains positive (although more uncertain). So why does body mass go negative in the joint model? Well to figure that out, we can look at the pointwise WAIC predictions between the Smart model and the joint model:

In [ ]:
# Full model pointwise WAIC estimates
pWAIC_mb <- WAIC_mb$pointwise[, "waic"]
# Brain size model pointwise WAIC estimates
pWAIC_b <- WAIC_b$pointwise[, "waic"]
# Difference between them
dWAIC <- pWAIC_mb - pWAIC_b

In [ ]:
Rplot <- R
Rplot[Rplot < 0] <- min(Rplot[Rplot > 0])

In [ ]:
# Set figure size
options(repr.plot.width = 10, repr.plot.height = 5)

# Select species to label
sppx <- c('Cebus albifrons', 'Cebus capucinus', 'Cebus olivaceus',
          'Gorilla gorilla', 'Lepilemur leucopus', 'Cacajao melanocephalus')
sindx <- match(sppx, mdata$Spp)

plot(dWAIC, L, cex = sqrt(50 + Rplot*30)/7, pch = 21, lwd = 0.5,
     bg = adjustcolor('blue', 0.3), col = 'black', ylim = range(c(L, -3)),
     xlim = range(dWAIC) + c(-0.05, 0.3)*diff(range(dWAIC)),
     xlab = 'Pointwise difference in WAIC', ylab = 'log(longevity) (std)', cex.lab = 1.3)
text(dWAIC[sindx], L[sindx], mdata$Spp[sindx], cex = 0.8,
     pos = ifelse(dWAIC[sindx] > 0.5*max(dWAIC), 2, 4))
text(-0.01, -2.9, '<-- Full model better', adj = 1, font = 2)
text(0.01, -2.9, 'Smart model better -->', adj = 0, font = 2)
abline(v = 0, lty = 3)
abline(h = 0, lty = 3)
# savefig('waicprime.jpg', width = 10, height = 5)

# Conditionaily

A couple of classic examples of conditioning on a collider can be seen in thinking about manatees. Because of the observation of scars on so many manatees, and so many dying from boat strikes, Florida passed laws requring cowlings over the propellers of speadboats to save manatees. However the problem with this - other than helping with nasty scars - is that it doesn't deal with the cause of death in manatees, which is idiots running them over in shallow water and causing massive internal trauma. In other words 


A second favourite example comes from WWII, where [Abraham Wald](https://en.wikipedia.org/wiki/Abraham_Wald) was tasked with figuring out where to place the limited armour that could be added to bombers and other allied airplanes. In [a series of memos](http://people.ucsc.edu/~msmangel/Wald.pdf) Wald figured out that rather than putting armour on areas of returning planes that had lots of bullet holes, it was correct to put the armour on areas that did not. As for the manatees, by conditioning on survivors, the cause of death becomes obscured. 

# Interactions

Interactions measure the influence of various predictors conditional on the other predictors in a model. Interactions can happen with the slope or the intercept or both. 


Let's start with a discrete interaction on a continuous variable. By way of example, let's have a look at the ruggedness of African nations:

In [ ]:
adata <- read.csv('rugged.csv', sep = ";")
adata <- adata[!is.na(adata$rgdppc_2000), ]
adata$Continent <- c('Non-African', 'African')[adata$cont_africa + 1]
head(adata)

In [ ]:
names(adata)

If we take a look at the economic output of each country (measured by GDP, which is problematic, but anyhow...), the relationship between the ruggedness of the terrain and GDP is positive among African nations while being negative everywhere else

In [ ]:
# GDP ratio relative to mean
GDP <- log(adata$rgdppc_2000)/mean(log(adata$rgdppc_2000))
# Ruggedness
RUG <- adata$rugged/max(adata$rugged)
meanRUG <- mean(RUG)
# Africa index - equivlant to indexall(adata$Continent)
Continent <- levels(factor(adata$Continent))
Ia <- as.integer(factor(adata$Continent))

In [ ]:
Continent

In [ ]:
# Base R has lm() built in to fit a line

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
par(mfrow = c(1, 2))

# New data
xnew <- seq(0, 1, length.out = 30)
# Fit with lm
cf <- coef(lm(GDP[Ia == 1] ~ RUG[Ia == 1]))

plot(RUG[Ia == 1], GDP[Ia == 1], col = 'dodgerblue', xlim = c(0, 1), ylim = c(0.7, 1.35),
     main = 'African nations', xlab = 'Ruggedness (ratio)',
     ylab = 'log(GDP) (ratio to mean)', cex.main = 1.3, cex.lab = 1.2)
lines(xnew, cf[1] + cf[2]*xnew, col = 'dodgerblue')

# Fit with lm
cf <- coef(lm(GDP[Ia == 2] ~ RUG[Ia == 2]))

plot(RUG[Ia == 2], GDP[Ia == 2], col = 'dodgerblue', xlim = c(0, 1), ylim = c(0.7, 1.35),
     main = 'Non-African nations', xlab = 'Ruggedness (ratio)',
     ylab = '', cex.main = 1.3, cex.lab = 1.2)
lines(xnew, cf[1] + cf[2]*xnew, col = 'dodgerblue')

par(mfrow = c(1, 1))
# savefig('rugged.jpg', width = 10, height = 4)

So to put together an interaction model we can start by guessing at some relatively benign priors and adding them to a model that allows for different intercepts and slopes for African/non-African nations:

He writes this model out three times in a row, changing only how tight the priors are, so here it is compiled once with the two prior standard deviations (`sd_int`, `sd_rug`) passed in as data. Each of the three attempts below just sets those two numbers.

In [ ]:
# Ruggedness model
rugged_code <- "
data {
  int<lower=0> N;
  int<lower=1> Ncont;
  array[N] int<lower=1> Ia;
  vector[N] RUG;
  vector[N] GDP;
  real<lower=0> sd_int;             // Prior SD on the intercepts
  real<lower=0> sd_rug;             // Prior SD on the ruggedness effects
}
parameters {
  vector[Ncont] Intercept;          // African/non intercepts
  vector[Ncont] Ruggedness;         // Ruggedness effect
  real<lower=0> SD_obs;             // Error
}
transformed parameters {
  // Linear model
  vector[N] mu = Intercept[Ia] + Ruggedness[Ia] .* RUG;
}
model {
  // Priors
  Intercept ~ normal(1, sd_int);
  Ruggedness ~ normal(0, sd_rug);
  SD_obs ~ exponential(1);

  // Likelihood
  GDP ~ normal(mu, SD_obs);
}
"
rugged <- cmdstan_model(write_stan_file(rugged_code))

# First guess at the priors
sd_int <- 1
sd_rug <- 1

In [ ]:
# Sample from prior predictive distribution
ppd_ <- data.frame(Intercept = rnorm(200, 1, sd_int), Ruggedness = rnorm(200, 0, sd_rug))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)

# Prior predictive for the two continents, which have the same (independent) priors
plot_rugged_ppd <- function(pp, alpha = 0.3){
    par(mfrow = c(1, 2))
    # New data
    xnew <- seq(0, 1, length.out = 30)

    plot(RUG[Ia == 1], GDP[Ia == 1], col = 'dodgerblue', xlim = c(0, 1), ylim = c(0.7, 1.35),
         main = 'African nations', xlab = 'Ruggedness (ratio)',
         ylab = 'log(GDP) (ratio to mean)', cex.main = 1.3, cex.lab = 1.2)
    # Sample from prior predictive
    for (i in 1:nrow(pp)) lines(xnew, pp$Intercept[i] + pp$Ruggedness[i]*xnew,
                                col = adjustcolor('black', alpha))

    plot(RUG[Ia == 2], GDP[Ia == 2], col = 'dodgerblue', xlim = c(0, 1), ylim = c(0.7, 1.35),
         main = 'Non-African nations', xlab = 'Ruggedness (ratio)',
         ylab = '', cex.main = 1.3, cex.lab = 1.2)
    # Sample from prior predictive
    for (i in 1:nrow(pp)) lines(xnew, pp$Intercept[i] + pp$Ruggedness[i]*xnew,
                                col = adjustcolor('black', alpha))
    par(mfrow = c(1, 1))
}
plot_rugged_ppd(ppd_)
# savefig('rugged_ppd.jpg', width = 10, height = 4)

Well this is a mess, let's try something tighter:

In [ ]:
# Ruggedness model - tighter priors
sd_int <- 0.1
sd_rug <- 0.1

In [ ]:
# Sample from prior predictive distribution
ppd_ <- data.frame(Intercept = rnorm(200, 1, sd_int), Ruggedness = rnorm(200, 0, sd_rug))

In [ ]:
plot_rugged_ppd(ppd_)
# savefig('rugged_ppd2.jpg', width = 10, height = 4)

Humm, not bad, maybe a little too tight (missing stuff at the bottom), one more try:

In [ ]:
# Ruggedness model - a wider prior on the ruggedness effect
sd_int <- 0.1
sd_rug <- 0.3

In [ ]:
# Sample from prior predictive distribution
ppd_ <- data.frame(Intercept = rnorm(200, 1, sd_int), Ruggedness = rnorm(200, 0, sd_rug))

In [ ]:
plot_rugged_ppd(ppd_)
# savefig('rugged_ppd3.jpg', width = 10, height = 4)

Ok, seems to get the balance of just a bit wider than is sensible. So let's put some data in and see what we get

In [ ]:
trace_r <- sample_stan(rugged, list(N = length(GDP), Ncont = length(Continent), Ia = Ia,
                                    RUG = RUG, GDP = GDP, sd_int = sd_int, sd_rug = sd_rug))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)

# Posterior densities, with the continent indices spelled out
post_r <- trace_r$draws(c("Intercept", "Ruggedness", "SD_obs"), format = "matrix")
colnames(post_r) <- c(paste0('Intercept[', Continent, ']'),
                      paste0('Ruggedness[', Continent, ']'), 'SD_obs')

mcmc_areas(post_r, prob = 0.94, point_est = "mean")
# ggsave('rugged_posterior.jpg', dpi = 300)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
par(mfrow = c(1, 2))

# New data
xnew <- seq(0, 1, length.out = 30)
# Posterior draws for each continent: column 1 is African, column 2 is Non-African
b0_ <- trace_r$draws("Intercept", format = "matrix")
b1_ <- trace_r$draws("Ruggedness", format = "matrix")
# African draws
b0a <- as.vector(b0_[, 1]); b1a <- as.vector(b1_[, 1])
# Non-African draws
b0n <- as.vector(b0_[, 2]); b1n <- as.vector(b1_[, 2])
# One chain's worth of draws, as in the Python version
ix <- 1:1000

plot(RUG[Ia == 1], GDP[Ia == 1], col = 'dodgerblue', xlim = c(0, 1), ylim = c(0.7, 1.35),
     main = 'African nations', xlab = 'Ruggedness (ratio)',
     ylab = 'log(GDP) (ratio to mean)', cex.main = 1.3, cex.lab = 1.2)
# Sample from posterior
for (i in ix) lines(xnew, b0a[i] + b1a[i]*xnew, col = adjustcolor('black', 0.01))

plot(RUG[Ia == 2], GDP[Ia == 2], col = 'dodgerblue', xlim = c(0, 1), ylim = c(0.7, 1.35),
     main = 'Non-African nations', xlab = 'Ruggedness (ratio)',
     ylab = '', cex.main = 1.3, cex.lab = 1.2)
# Sample from posterior
for (i in ix) lines(xnew, b0n[i] + b1n[i]*xnew, col = adjustcolor('black', 0.01))

par(mfrow = c(1, 1))
# savefig('rugged_fits.jpg', width = 10, height = 4)

In [ ]:
# Net difference in ruggedness effect
Net_rug <- b1a - b1n

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
hist(Net_rug, col = 'dodgerblue', main = '',
     xlab = expression('Africa '*theta*' - Non-africa '*theta))
# savefig('rugged_net.jpg')

In [ ]:
1 - sum(Net_rug < 0)/length(Net_rug)

So what this says is a couple of things:

1. In flat places, GDP among African nations is a fraction (0.86) of flat Non-African (1.1) nations
2. As ruggedness increases, African nations become wealthier while Non-African nations become poorer
3. There is strong evidence of ruggedness having a more positive slope in African nations

Point (2) bears thinking about. Why does this happen? Well it has a very dark answer that you'll explore in the homework this week.

An important point about interactions is that they are ALWAYS difficult to interpret. One of the most basic problems is that they are statistically but not congnitively symmetric: 

    - *The effect of ruggedness on a nation's GDP depends on what continent it is from*
    - *The effect of continent depends on ruggedness*
    
If you read these carefully, one will make complete sense and the other is nonsense. This is your causal brain understanding that you can't move nations among continents.

# Bloomin' tulips

For a more comprehensive data, let's have a look at a continuous interaction, including combinations of water and shade on tulip blooming. 

First, let's import the data

In [ ]:
bdata <- read.csv('tulips.csv', sep = ";")
head(bdata)

And knowing our flower husbandry, we can readily imagine that the quantities of water and light, as well as their combination, influencing the blooming of flowers. First we standardize the variables:

In [ ]:
# Response scaled to proportion of max
B <- bdata$blooms/max(bdata$blooms)
# Water covariate - mean centred
W <- bdata$water - mean(bdata$water)
# Shade covariate - mean centred
S <- bdata$shade - mean(bdata$shade)

Next we need some priors - what would be sensible? Well now that blooms (B) has been scaled to be between 0 and 1, the intercept should be mostly in this range. To figure out what's within that range, we can first take a stab that the intercept will be at roughly 0.5 when water (W) and shade (S) are at their mean values. For the standard deviation we can then figure out what the cumulative probability is for 0 and 1, given a $N(0.5, \sigma)$ distribtuion, with $\sigma$ found by guessing:

In [ ]:
# Start with Normal(0.5, 1) at zero
pnorm(0, 0.5, 1)

Well this is too big, more than 30% of the prior would be below zero, maybe we should aim for the conventional 5% (which means 2.5% should be below zero and the other 2.5% above 1). How about $N(0.5, 0.25)$?

In [ ]:
# Normal(0.5, 0.25) at zero
pnorm(0, 0.5, 0.25)

In [ ]:
# Normal(0.5, 0.25) at one
1 - pnorm(1, 0.5, 0.25)

Looking good - ok onto the slopes. What's reasonable here? Well despite knowing there's likely an interaction effect, our ignorant, slightly wide prior should allow for either water or shade to account for the full 0 to 1 range in blooms. So to do this we can see how wide the range is for each (using `diff(range())` in R, the equivalent of the funny `np.ptp()` function in `numpy`:

In [ ]:
# Peak to peak range, the name of which comes from min/max calculations of waveforms
c(diff(range(W)), diff(range(S)))

So over a range of 2 units, we need a prior that can span the full range from 0 to 1. This would imply a slope of -0.5 or 0.5 (i.e. $2*0.5=1$). So a sensible prior for $2SD=0.5$ would seem to be 0.25:

In [ ]:
# Normal(0, 0.25) at -0.5
pnorm(-0.5, 0, 0.25)

In [ ]:
# Normal(0, 0.25) at 0.5
1 - pnorm(0.5, 0, 0.25)

Perfect, now putting this all together:

In [ ]:
# Basic bloom model
bloom_code <- "
data {
  int<lower=0> N;
  vector[N] water;
  vector[N] shade;
  vector[N] blooms;
}
parameters {
  real Intercept;         // African/non intercepts
  real Water;             // Water effect
  real Shade;             // Shade effect
  real<lower=0> SD_obs;   // Error
}
transformed parameters {
  // Linear model
  vector[N] mu = Intercept + Water*water + Shade*shade;
}
model {
  // Priors
  Intercept ~ normal(0.5, 0.25);
  Water ~ normal(0, 0.25);
  Shade ~ normal(0, 0.25);
  SD_obs ~ exponential(1);

  // Likelihood
  blooms ~ normal(mu, SD_obs);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = normal_lpdf(blooms[i] | mu[i], SD_obs);
}
"
bloom <- cmdstan_model(write_stan_file(bloom_code))

In [ ]:
# Interaction bloom model - the slope prior SD is passed in as data, since he tightens it below
ibloom_code <- "
data {
  int<lower=0> N;
  vector[N] water;
  vector[N] shade;
  vector[N] blooms;
  real<lower=0> sd_slope;   // Prior SD on the three slopes
}
parameters {
  real Intercept;         // African/non intercepts
  real Water;             // Water effect
  real Shade;             // Shade effect
  real ShadeWater;        // Interaction effect
  real<lower=0> SD_obs;   // Error
}
transformed parameters {
  // Linear model
  vector[N] mu = Intercept + Water*water + Shade*shade + ShadeWater*shade .* water;
}
model {
  // Priors
  Intercept ~ normal(0.5, 0.25);
  Water ~ normal(0, sd_slope);
  Shade ~ normal(0, sd_slope);
  ShadeWater ~ normal(0, sd_slope);
  SD_obs ~ exponential(1);

  // Likelihood
  blooms ~ normal(mu, SD_obs);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = normal_lpdf(blooms[i] | mu[i], SD_obs);
}
"
ibloom <- cmdstan_model(write_stan_file(ibloom_code))

# First guess at the slope priors
sd_slope <- 0.25

In [ ]:
# Sample from prior predictive distributions
ppd_ib <- data.frame(Intercept = rnorm(200, 0.5, 0.25), Water = rnorm(200, 0, sd_slope),
                     Shade = rnorm(200, 0, sd_slope), ShadeWater = rnorm(200, 0, sd_slope))

In [ ]:
dim(ppd_ib)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 4)

# Blooms against water at three levels of shade
plot_tulip_lines <- function(pp, inter = TRUE, data = FALSE, prefix = '', alpha = 0.3){
    # New data
    xnew <- seq(-1, 1, length.out = 30)
    for (sx in c(-1, 0, 1)){
        plot(NA, xlim = c(-1, 1), ylim = c(0, 1), main = paste0(prefix, 'Shade=', sx),
             xlab = 'Water', ylab = 'Blooms (scaled)', cex.main = 1.3, cex.lab = 1.2)
        for (i in 1:nrow(pp)){
            mu_ <- pp$Intercept[i] + pp$Water[i]*xnew + pp$Shade[i]*sx
            if (inter) mu_ <- mu_ + pp$ShadeWater[i]*xnew*sx
            lines(xnew, mu_, col = adjustcolor('black', alpha))
        }
        if (data) points(W[S == sx], B[S == sx], col = 'dodgerblue')
    }
}
par(mfrow = c(1, 3))
plot_tulip_lines(ppd_ib)
par(mfrow = c(1, 1))

Well these look like shit, maybe something a bit smaller?

In [ ]:
# Interaction bloom model - smaller slope priors
sd_slope <- 0.1

In [ ]:
# Sample from prior predictive distributions
ppd_ib <- data.frame(Intercept = rnorm(200, 0.5, 0.25), Water = rnorm(200, 0, sd_slope),
                     Shade = rnorm(200, 0, sd_slope), ShadeWater = rnorm(200, 0, sd_slope))

In [ ]:
par(mfrow = c(1, 3))
plot_tulip_lines(ppd_ib)
par(mfrow = c(1, 1))
# savefig('tulip_ppd.jpg', width = 15, height = 4)

These seem a bit better, let's run with them

In [ ]:
bdat <- list(N = length(B), water = W, shade = S, blooms = B)
trace_b <- sample_stan(bloom, bdat)
trace_ib <- sample_stan(ibloom, c(bdat, list(sd_slope = sd_slope)))

In [ ]:
trace_b$summary(c("Intercept", "Water", "Shade", "SD_obs"))

In [ ]:
trace_ib$summary(c("Intercept", "Water", "Shade", "ShadeWater", "SD_obs"))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)

# Number of samples to take
ns <- 20
# Grab ns random samples
post_b <- as.data.frame(trace_b$draws(c("Intercept", "Water", "Shade"), format = "matrix"))
post_ib <- as.data.frame(trace_ib$draws(c("Intercept", "Water", "Shade", "ShadeWater"), format = "matrix"))
Ix <- sample(nrow(post_ib), ns)

par(mfrow = c(2, 3))
# Basic model - top row
plot_tulip_lines(post_b[Ix, ], inter = FALSE, data = TRUE, prefix = 'Basic model: ')
# Interaction model - bottom row
plot_tulip_lines(post_ib[Ix, ], inter = TRUE, data = TRUE, prefix = 'Interaction model: ')
par(mfrow = c(1, 1))
# savefig('tulip_fits.jpg', width = 15, height = 10)

So, with just one interaction we've got a heck of a lot going on. Nervous yet? 